# Level 1: one game, minute by minute

Level 0 looked at the tables, this level follows one game: the economy and the army of both
players over game time, the supply blocks, the build order, the upgrades, the fights and where
they happened. The questions are the ones a coach asks of a replay, the measures are the ones
the community uses (the [spending quotient](https://liquipedia.net/starcraft2/Spending_quotient),
the workers and the army value at fixed minutes, the commands per minute).

Every question is answered twice, with [polars](https://docs.pola.rs/) expressions and with
[DuckDB](https://duckdb.org/docs/) SQL over the same files; `s2.same()` checks that both return the
same rows, so each pair is a worked translation between the two.

Previous: [Level 0](./Level_0_Inspect.ipynb). Next: [Level 2](./Level_2_Over_Time.ipynb), the same
measures over hundreds of games and the months they were played.

In [1]:
import polars as pl
import plotly.express as px

import snapshot as s2
import viz

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

# The snapshot: `$S2_IPC_DIR`, or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()
# polars reads the files lazily, one LazyFrame per file.
details = s2.scan(IPC_DIR, "details")
stats = s2.scan(IPC_DIR, "stats")
upgrades = s2.scan(IPC_DIR, "upgrades")
unit_born = s2.scan(IPC_DIR, "unit_born")
unit_died = s2.scan(IPC_DIR, "unit_died")
# DuckDB reads the same files as SQL views of the same names, plus `players` and `games_1v1`.
con = s2.duck(IPC_DIR)

# One row per player of every game with its tracker player_id, and the 1v1 games with the matchup.
# They read every unit_born file, so they are collected once and reused.
players = s2.players(IPC_DIR).collect().lazy()
games_1v1 = s2.one_vs_one(IPC_DIR).collect().lazy()

## Pick a game

The latest 1v1 of the player with the most games in the snapshot: on a snapshot of your own
replays it is your latest game. `GAME` is its `ext_fs_id`, change it to follow another one.

In [2]:
most_played = (
    players.group_by("player")
    .agg(pl.len().alias("games"))
    .sort("games", "player", descending=[True, False])
    .head(1)
    .collect()
)
PLAYER = most_played["player"][0]
game = (
    games_1v1.filter(pl.col("player") == PLAYER)
    .sort("ext_datetime", descending=True)
    .head(1)
    .select("ext_fs_id", "ext_datetime", "title", "player", "race", "player_result", "opponent", "opponent_race", "matchup")
    .collect()
)
GAME = game["ext_fs_id"][0]
TITLE = f"{game['player'][0]} ({game['race'][0]}) vs {game['opponent'][0]} ({game['opponent_race'][0]}) on {game['title'][0]}"
print(TITLE)
game

Sazed (Z) vs FirosAryn (T) on Persephone LE


ext_fs_id,ext_datetime,title,player,race,player_result,opponent,opponent_race,matchup
u64,datetime[ns],str,str,str,str,str,str,str
34904,2025-11-03 19:01:16.234195,"""Persephone LE""","""Sazed""","""Z""","""Loss""","""FirosAryn""","""T""","""ZvT"""


In [3]:
# `?` is a parameter, the value is passed apart from the query, so a name with a quote is no problem.
sql_player = con.sql("SELECT player, count(*) AS games FROM players GROUP BY player ORDER BY games DESC, player LIMIT 1")
sql_game = con.sql(
    """
    SELECT ext_fs_id, ext_datetime, title, player, race, player_result, opponent, opponent_race, matchup
    FROM games_1v1
    WHERE player = ?
    ORDER BY ext_datetime DESC
    LIMIT 1
    """,
    params=[PLAYER],
)
print(s2.same(most_played, sql_player), "|", s2.same(game, sql_game))

same 1 rows, 2 columns | same 1 rows, 9 columns


## The two sides

The tracker events identify the players by `player_id`, the commands by `player_name`: `sides`
has both for the two players, every query below joins it. DuckDB sees the polars frames of the
notebook by their variable names, `sides` is a table in the SQL twins.

In [4]:
sides = (
    games_1v1.filter(pl.col("ext_fs_id") == GAME)
    .select("player_id", "player_name", "player", "race", "player_result")
    .sort("player_id")
    .collect()
)
# The color of each player in the charts of this notebook, the first two categorical slots.
PLAYER_COLORS = {player: viz.SERIES[i] for i, player in enumerate(sides["player"])}
sides

player_id,player_name,player,race,player_result
u8,str,str,str,str
1,"""&lt;chezs&gt;<sp/>Sazed""","""Sazed""","""Z""","""Loss"""
2,"""FirosAryn""","""FirosAryn""","""T""","""Win"""


In [5]:
s2.same(sides, con.sql("SELECT player_id, player_name, player, race, player_result FROM games_1v1 WHERE ext_fs_id = ?", params=[GAME]))

'same 2 rows, 5 columns'

## The economy and the army over time

The `stats` are the PlayerStats tracker events, one sample per player every 160 game loops (about
7 seconds): the workers mining, the collection rate (the income, minerals and vespene per minute),
the resources unspent, the value of the army alive and the supply. `s2.game_seconds()` turns the
game loops into the seconds of the in-game clock, the `game_seconds` macro does it in SQL.

In [6]:
economy = (
    stats.filter(pl.col("ext_fs_id") == GAME)
    .join(sides.lazy().select("player_id", "player", "race"), on="player_id")
    .select(
        "player",
        "race",
        (s2.game_seconds() / 60).alias("minute"),
        pl.col("workers_active_count").alias("workers"),
        (pl.col("minerals_collection_rate") + pl.col("vespene_collection_rate")).alias("income"),
        (pl.col("minerals_current") + pl.col("vespene_current")).alias("unspent"),
        (pl.col("minerals_used_current_army") + pl.col("vespene_used_current_army")).alias("army"),
        pl.col("food_used").alias("supply"),
        pl.col("food_made").alias("supply_cap"),
    )
    .sort("player", "minute")
    .collect()
)
economy.head(6)

player,race,minute,workers,income,unspent,army,supply,supply_cap
str,str,f64,i32,i32,i32,i32,i32,i32
"""FirosAryn""","""T""",0.000744,12,0,50,0,12,15
"""FirosAryn""","""T""",0.119048,12,251,30,0,13,15
"""FirosAryn""","""T""",0.238095,13,671,70,0,14,15
"""FirosAryn""","""T""",0.357143,13,727,0,0,14,15
"""FirosAryn""","""T""",0.47619,14,755,90,0,15,15
"""FirosAryn""","""T""",0.595238,14,811,140,0,15,15


In [7]:
sql = con.sql(
    """
    SELECT
        p.player,
        p.race,
        game_seconds(s.ext_replay_loop) / 60 AS minute,
        s.workers_active_count AS workers,
        s.minerals_collection_rate + s.vespene_collection_rate AS income,
        s.minerals_current + s.vespene_current AS unspent,
        s.minerals_used_current_army + s.vespene_used_current_army AS army,
        s.food_used AS supply,
        s.food_made AS supply_cap
    FROM stats s
    JOIN sides p USING (player_id)
    WHERE s.ext_fs_id = ?
    """,
    params=[GAME],
)
s2.same(economy, sql)

'same 351 rows, 9 columns'

Four measures, one panel each, the two players in the same two colors in every chart of this
notebook. `unpivot` turns the four columns into one long frame, the format plotly express facets
by.

In [8]:
long = economy.unpivot(index=["player", "minute"], on=["workers", "income", "army", "unspent"], variable_name="measure", value_name="value")
fig = px.line(
    long, x="minute", y="value", color="player", facet_col="measure", facet_col_wrap=2,
    color_discrete_map=PLAYER_COLORS, category_orders={"measure": ["workers", "income", "army", "unspent"]},
    title=f"{TITLE}: the economy and the army, minute by minute", height=620, facet_col_spacing=0.07,
)
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
viz.show(fig, "level_1_economy")

## Supply blocks

A player is supply blocked when the supply used reaches the supply made before the maximum of
200: every sample where it happens is about 7 seconds without production. The first block is
usually the opening, the ones after are the ones to fix.

In [9]:
blocked = (
    economy.lazy()
    .filter((pl.col("supply") >= pl.col("supply_cap")) & (pl.col("supply_cap") < 200))
    .group_by("player")
    .agg(
        pl.len().alias("blocked_samples"),
        pl.col("minute").min().round(1).alias("first_block_minute"),
        pl.col("minute").max().round(1).alias("last_block_minute"),
    )
    .with_columns((pl.col("blocked_samples") * s2.STATS_SAMPLE_SECONDS).round(0).alias("blocked_seconds"))
    .sort("player")
    .collect()
)
blocked

player,blocked_samples,first_block_minute,last_block_minute,blocked_seconds
str,u32,f64,f64,f64
"""FirosAryn""",21,0.5,17.0,150.0
"""Sazed""",5,0.4,9.5,36.0


In [10]:
sql = con.sql(f"""
    SELECT
        player,
        count(*) AS blocked_samples,
        round(min(minute), 1) AS first_block_minute,
        round(max(minute), 1) AS last_block_minute,
        round(count(*) * {s2.STATS_SAMPLE_SECONDS}, 0) AS blocked_seconds
    FROM economy
    WHERE supply >= supply_cap AND supply_cap < 200
    GROUP BY player
""")
s2.same(blocked, sql)

'same 2 rows, 5 columns'

## Spending

The [spending quotient](https://liquipedia.net/starcraft2/Spending_quotient) of TeamLiquid
scores how well the income is spent, from the average income `i` and the average unspent
resources `u`: `SQ = 35 (0.00137 i − ln u) + 240`. A grandmaster is often above 90, the score
is not meaningful under 600 income per minute, so the first two minutes are left out, and a
player banking for a warp-in or at 200 supply is penalized.

In [11]:
spending = (
    economy.lazy()
    .filter(pl.col("minute") >= 2)
    .group_by("player")
    .agg(
        pl.col("income").mean().alias("avg_income"),
        pl.col("unspent").mean().alias("avg_unspent"),
        pl.col("workers").max().alias("max_workers"),
        pl.col("supply").max().alias("max_supply"),
    )
    .with_columns((35 * (0.00137 * pl.col("avg_income") - pl.col("avg_unspent").log()) + 240).alias("spending_quotient"))
    .sort("player")
    .collect()
)
spending

player,avg_income,avg_unspent,max_workers,max_supply,spending_quotient
str,f64,f64,i32,i32,f64
"""FirosAryn""",3159.759494,4857.436709,95,200,94.421152
"""Sazed""",3017.471698,4022.396226,79,200,94.20061


In [12]:
sql = con.sql("""
    SELECT
        player,
        avg(income) AS avg_income,
        avg(unspent) AS avg_unspent,
        max(workers) AS max_workers,
        max(supply) AS max_supply,
        35 * (0.00137 * avg(income) - ln(avg(unspent))) + 240 AS spending_quotient
    FROM economy
    WHERE minute >= 2
    GROUP BY player
""")
s2.same(spending, sql)

'same 2 rows, 6 columns'

## The build order

The structures each player started, in order, from the `unit_born` events: a structure is born
when it is placed. `s2.STRUCTURES` lists them by race, the morphs (Lair, Orbital Command, Warp
Gate) are born from another structure and are not in it. The ordinal of each structure per player
is a window: `int_range().over()` in polars, `row_number() OVER (PARTITION BY ...)` in SQL.

In [13]:
build = (
    unit_born.filter((pl.col("ext_fs_id") == GAME) & (pl.col("ext_replay_loop") > 0) & s2.is_structure())
    .join(sides.lazy().select(pl.col("player_id").alias("control_player_id"), "player"), on="control_player_id")
    .sort("ext_replay_loop", "unit_tag_index")
    .select("player", pl.col("unit_type_name").alias("structure"), s2.game_seconds().floor().alias("seconds"))
    .with_columns((pl.int_range(pl.len()) + 1).over("player").alias("n"))
    .filter(pl.col("n") <= 15)
    .collect()
)
# Side by side: one column per player, `m:ss structure`.
build.with_columns(
    pl.format("{}:{} {}", (pl.col("seconds") // 60).cast(pl.Int32), (pl.col("seconds") % 60).cast(pl.Int32).cast(pl.String).str.zfill(2), "structure").alias("entry")
).pivot(on="player", index="n", values="entry")

n,FirosAryn,Sazed
i64,str,str
1,"""0:38 SupplyDepot""","""1:31 Extractor"""
2,"""1:42 Refinery""","""2:00 Hatchery"""
3,"""1:58 Refinery""","""2:02 SpawningPool"""
4,"""1:58 Barracks""","""3:48 Hatchery"""
5,"""2:06 SupplyDepot""","""4:22 EvolutionChamber"""
6,"""2:09 SupplyDepot""","""4:23 Extractor"""
7,"""2:10 CommandCenter""","""4:24 Extractor"""
8,"""3:07 BarracksReactor""","""4:35 RoachWarren"""
9,"""3:13 Factory""","""4:53 Hatchery"""


In [14]:
sql = con.sql(f"""
    SELECT
        p.player,
        b.unit_type_name AS structure,
        floor(game_seconds(b.ext_replay_loop)) AS seconds,
        row_number() OVER (PARTITION BY p.player ORDER BY b.ext_replay_loop, b.unit_tag_index) AS n
    FROM unit_born b
    JOIN sides p ON p.player_id = b.control_player_id
    WHERE b.ext_fs_id = ? AND b.ext_replay_loop > 0 AND b.unit_type_name IN ({s2.sql_list(s2.ALL_STRUCTURES)})
    QUALIFY n <= 15
""", params=[GAME])
s2.same(build, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 30 rows, 4 columns'

## The upgrades

When each upgrade finished, from the `upgrades` events. The sprays and the rewards are upgrades
too in the tracker, `s2.is_cosmetic_upgrade()` removes them (a regular expression on the same
prefixes in SQL).

In [15]:
researched = (
    upgrades.filter((pl.col("ext_fs_id") == GAME) & (pl.col("ext_replay_loop") > 0) & ~s2.is_cosmetic_upgrade())
    .join(sides.lazy().select("player_id", "player"), on="player_id")
    .select("player", pl.col("name").alias("upgrade"), s2.game_seconds().floor().alias("seconds"))
    .sort("seconds", "player")
    .collect()
)
researched

player,upgrade,seconds
str,str,f64
"""Sazed""","""zerglingmovementspeed""",238.0
"""FirosAryn""","""HighCapacityBarrels""",365.0
"""FirosAryn""","""Stimpack""",507.0
"""FirosAryn""","""TerranInfantryWeaponsLevel1""",516.0
"""FirosAryn""","""TerranInfantryArmorsLevel1""",517.0
"""FirosAryn""","""TerranVehicleWeaponsLevel1""",518.0
"""Sazed""","""Burrow""",603.0
"""FirosAryn""","""ShieldWall""",660.0
"""FirosAryn""","""DrillClaws""",662.0


In [16]:
COSMETIC = "^(" + "|".join(s2.COSMETIC_UPGRADE_PREFIXES) + ")"
sql = con.sql(
    """
    SELECT p.player, u.name AS upgrade, floor(game_seconds(u.ext_replay_loop)) AS seconds
    FROM upgrades u
    JOIN sides p USING (player_id)
    WHERE u.ext_fs_id = ? AND u.ext_replay_loop > 0 AND NOT regexp_matches(u.name, ?)
    """,
    params=[GAME, COSMETIC],
)
s2.same(researched, sql)

'same 29 rows, 3 columns'

## The fights

`minerals_lost_army` and `vespene_lost_army` count the value of the army lost so far, so the
difference between two minutes is the army lost in that minute: a fight is a minute where both
players lose a lot, a harassment one where only one does. The difference with the previous row
of the same player is `shift(1).over("player")` in polars, `lag() OVER (PARTITION BY player)` in
SQL.

In [17]:
lost = (
    stats.filter(pl.col("ext_fs_id") == GAME)
    .join(sides.lazy().select("player_id", "player"), on="player_id")
    .select(
        "player",
        (s2.game_seconds() // 60).cast(pl.Int32).alias("minute"),
        (pl.col("minerals_lost_army") + pl.col("vespene_lost_army")).alias("lost_so_far"),
    )
    .group_by("player", "minute")
    .agg(pl.col("lost_so_far").max())
    .sort("player", "minute")
    .with_columns((pl.col("lost_so_far") - pl.col("lost_so_far").shift(1).over("player").fill_null(0)).alias("army_lost"))
    .select("player", "minute", "army_lost")
    .collect()
)
lost.sort("army_lost", descending=True).head(5)

player,minute,army_lost
str,i32,i32
"""Sazed""",19,7984
"""Sazed""",20,3850
"""Sazed""",13,3224
"""Sazed""",15,2700
"""FirosAryn""",20,2500


In [18]:
sql = con.sql(
    """
    WITH per_minute AS (
        SELECT
            p.player,
            floor(game_seconds(s.ext_replay_loop) / 60)::INTEGER AS minute,
            max(s.minerals_lost_army + s.vespene_lost_army) AS lost_so_far
        FROM stats s
        JOIN sides p USING (player_id)
        WHERE s.ext_fs_id = ?
        GROUP BY ALL
    )
    SELECT player, minute, lost_so_far - lag(lost_so_far, 1, 0) OVER (PARTITION BY player ORDER BY minute) AS army_lost
    FROM per_minute
    """,
    params=[GAME],
)
s2.same(lost, sql)

'same 42 rows, 3 columns'

In [19]:
fig = px.bar(
    lost, x="minute", y="army_lost", color="player", barmode="group", color_discrete_map=PLAYER_COLORS,
    title=f"{TITLE}: army value lost per minute", labels={"army_lost": "army value lost (minerals + vespene)"},
)
viz.show(fig, "level_1_army_lost")

## Who killed what

The `unit_died` events name the unit and, when a player killed it, the killer unit and the
killer player. The units that expire (a Larva morphed, a Force Field) have no killer. The top
pairs of the game:

In [20]:
kills = (
    unit_died.filter((pl.col("ext_fs_id") == GAME) & (pl.col("unit_killer_name") != "") & pl.col("killer_player_id").is_not_null())
    .join(sides.lazy().select(pl.col("player_id").alias("killer_player_id"), pl.col("player").alias("killer")), on="killer_player_id")
    .group_by("killer", pl.col("unit_killer_name").alias("killer_unit"), pl.col("unit_died_name").alias("victim"))
    .agg(pl.len().alias("kills"))
    .collect()
)
kills.sort("kills", "killer", "killer_unit", "victim", descending=[True, False, False, False]).head(12)

killer,killer_unit,victim,kills
str,str,str,u32
"""FirosAryn""","""Marine""","""Broodling""",44
"""FirosAryn""","""Hellion""","""Zergling""",37
"""FirosAryn""","""OrbitalCommand""","""MULE""",27
"""FirosAryn""","""Ghost""","""Broodling""",21
"""FirosAryn""","""Ghost""","""Zergling""",17
"""FirosAryn""","""Marine""","""Roach""",15
"""Sazed""","""Ravager""","""Marine""",14
"""Sazed""","""BroodlingEscort""","""Marine""",12
"""Sazed""","""Roach""","""Marine""",12


In [21]:
sql = con.sql(
    """
    SELECT p.player AS killer, d.unit_killer_name AS killer_unit, d.unit_died_name AS victim, count(*) AS kills
    FROM unit_died d
    JOIN sides p ON p.player_id = d.killer_player_id
    WHERE d.ext_fs_id = ? AND d.unit_killer_name <> ''
    GROUP BY ALL
    """,
    params=[GAME],
)
s2.same(kills, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 110 rows, 4 columns'

## Where the units died

The map: the position of every unit killed by a player, colored by who killed it, with the
starting townhalls of both players as squares. The units that die on their own (the Larva, the
Broodlings, the Invisible Target Dummies that the Lurkers attack) are left out.

In [22]:
import plotly.graph_objects as go

EXPIRES = ["InvisibleTargetDummy", "Larva", "Egg", "Broodling", "BroodlingEscort", "AdeptPhaseShift", "ForceField", "KD8Charge", "CreepTumorBurrowed"]
deaths = (
    unit_died.filter((pl.col("ext_fs_id") == GAME) & pl.col("killer_player_id").is_not_null() & ~pl.col("unit_died_name").is_in(EXPIRES))
    .join(sides.lazy().select(pl.col("player_id").alias("killer_player_id"), pl.col("player").alias("killed_by")), on="killer_player_id")
    .select("killed_by", "x", "y", pl.col("unit_died_name").alias("unit"), s2.game_seconds().floor().alias("seconds"))
    .collect()
)
starts = (
    unit_born.filter((pl.col("ext_fs_id") == GAME) & (pl.col("ext_replay_loop") == 0) & pl.col("unit_type_name").is_in(s2.TOWNHALLS))
    .join(sides.lazy().select(pl.col("player_id").alias("control_player_id"), "player"), on="control_player_id")
    .select("player", "x", "y")
    .collect()
)
fig = px.scatter(
    deaths, x="x", y="y", color="killed_by", hover_data=["unit", "seconds"], color_discrete_map=PLAYER_COLORS,
    title=f"{TITLE}: where the units died, by who killed them", width=700, height=650,
)
fig.update_traces(marker=dict(size=7, opacity=0.55))
for row in starts.iter_rows(named=True):
    fig.add_trace(go.Scatter(x=[row["x"]], y=[row["y"]], mode="markers", name=f"{row['player']} start",
                             marker=dict(symbol="square", size=16, color=PLAYER_COLORS[row["player"]], line=dict(width=2, color="#fcfcfb"))))
fig.update_yaxes(scaleanchor="x", scaleratio=1)
viz.show(fig, "level_1_deaths_map")
deaths.group_by("killed_by").agg(pl.len().alias("units_killed")).sort("killed_by")

killed_by,units_killed
str,u32
"""FirosAryn""",361
"""Sazed""",192


In [23]:
sql = con.sql(
    """
    SELECT p.player AS killed_by, count(*) AS units_killed
    FROM unit_died d
    JOIN sides p ON p.player_id = d.killer_player_id
    WHERE d.ext_fs_id = ? AND d.unit_died_name NOT IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY ALL
    """,
    params=[GAME, EXPIRES],
)
s2.same(deaths.group_by("killed_by").agg(pl.len().alias("units_killed")), sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 2 rows, 2 columns'

## Commands per minute

The ability commands of both players from the three `cmd_*` files, one row per command
(`s2.casts()`, the `casts` view): a Train, a Build, an attack order, an Inject. They are not
the APM of the game (the selections, the control groups and the camera are not in the snapshot),
they are the commands that act on the game, closer to the effective APM. The attack commands
apart, they show when the fights are.

In [24]:
commands = (
    s2.casts(IPC_DIR)
    .filter(pl.col("ext_fs_id") == GAME)
    .join(sides.lazy().select("player_name", "player"), on="player_name")
    .group_by("player", (s2.game_seconds() // 60).cast(pl.Int32).alias("minute"))
    .agg(pl.len().alias("commands"), (pl.col("ability_name") == "attack").sum().alias("attack_commands"))
    .sort("player", "minute")
    .collect()
)
commands.group_by("player").agg(pl.col("commands").mean().round(1).alias("commands_per_minute"), pl.col("attack_commands").mean().round(1).alias("attack_commands_per_minute")).sort("player")

player,commands_per_minute,attack_commands_per_minute
str,f64,f64
"""FirosAryn""",20.1,5.0
"""Sazed""",15.9,2.9


In [25]:
sql = con.sql(
    """
    SELECT
        p.player,
        floor(game_seconds(c.ext_replay_loop) / 60)::INTEGER AS minute,
        count(*) AS commands,
        count(*) FILTER (WHERE c.ability_name = 'attack') AS attack_commands
    FROM casts c
    JOIN sides p USING (player_name)
    WHERE c.ext_fs_id = ?
    GROUP BY ALL
    """,
    params=[GAME],
)
s2.same(commands, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 42 rows, 4 columns'

In [26]:
fig = px.line(
    commands, x="minute", y="commands", color="player", markers=True, color_discrete_map=PLAYER_COLORS,
    title=f"{TITLE}: ability commands per minute",
)
viz.show(fig, "level_1_commands_per_minute")

## What to take from Level 1

- One game is a set of time series per player: the `stats` every 7 seconds, the events (born,
  died, upgrades, commands) at their game loop. `s2.game_seconds()` puts them on the in-game clock.
- The measures of a game are differences and windows over those series: the army lost in a
  minute, the ordinal of a structure, the first supply block. `shift().over()`,
  `int_range().over()` and their SQL twins `lag() OVER`, `row_number() OVER`.
- Everything above was computed for one `ext_fs_id`; remove that filter and group by game and
  the same queries give the measures of every game, which is Level 2.

Next: [Level 2](./Level_2_Over_Time.ipynb).